### G EVAL

In [27]:
# !pip install deepeval

In [28]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import OllamaModel
import pandas as pd

C:\Users\rusha\AppData\Local\Temp\ipykernel_95852\2658974628.py:2: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


In [11]:
# 1. Create an OllamaModel instance first
ollama_model = OllamaModel(
    model="llama3.2",
    base_url="http://localhost:11434",  # Ollama default address
    temperature=0
)


In [93]:
# 1. Define the evaluation metric: check which of the required points the candidate mentioned
completeness = GEval(
    name="SelfIntroCompleteness",
    evaluation_steps=[
        "Check whether the candidate's answer mentions their full name.",
        "Check whether the candidate's answer mentions their educational background.",
        "Check whether the candidate's answer mentions their current job or work experience.",
        "Check whether the candidate's answer mentions at least one specific experience or skill.",
        "Score based on how many of the above points were covered: 4 points = 1.0, 3 points = 0.8, 2 points = 0.5, 1 point = 0.2, 0 points = 0.0."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],  # Only look at the answer, no expected_output needed
    model=ollama_model,  # Or any local model you have running in Ollama
    threshold=0.6
)

behavioral = GEval(
    name="STAR_Completeness",
    evaluation_steps=[
        "Check whether the candidate described a specific Situation or Task.",
        "Check whether the candidate explained the Action they personally took (not just 'we').",
        "Check whether the candidate stated the Result (ideally with some outcome or learning).",
        "The more complete and specific the answer, the higher the score; vague generic statements get lower scores."
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=ollama_model
)

situational = GEval(
    name="Situational_Reasoning",
    evaluation_steps=[
        "Check whether the candidate identified the key constraints or stakeholders in the scenario.",
        "Check whether the candidate proposed a concrete course of action (not just 'I would communicate').",
        "Check whether the candidate considered trade-offs or alternatives.",
        "Check whether the conclusion is actionable and realistic.",
        "Score: all four present = 1.0; missing trade-offs = 0.7; only vague principles = 0.3."
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=ollama_model
)

In [23]:

# 2. Construct a test case
test_case = LLMTestCase(
    input="Tell me about yourself.",
    actual_output="Hi, I'm Roshan.I graduated from Tribhuvan University with a degree in Computer Science. Currently I work as a backend developer at a fintech company, mainly using Python and Django."
)

# 3. Run the evaluation
completeness.measure(test_case)
print(completeness.score)   # 0-1
print(completeness.reason)  # Why this score was given

0.8
The response mentions the candidate's full name, educational background, and current job, but only briefly mentions their work experience and does not specify any particular experience or skill.


## IMPORTING THE CLASSIFIER MODEL

In [63]:
# =========================================================
# 11. INFERENCE
# =========================================================
from transformers import pipeline
base_dir = r"..\..\..\Extractor\classifier\question_classifier"
classifier = pipeline(
    "text-classification",
    model=base_dir,
    tokenizer=base_dir,
    device=-1,  # use 0 if you have a GPU
)

In [99]:
dataset_path = r"..\..\..\Docs\simple_questions.xlsx"

In [101]:
df = pd.read_excel(dataset_path).head()

In [102]:
df_test = df.head(5)

In [103]:
df_test

,question,role,ideal_answer,keywords
0,Tell me about a time you had to learn somethin...,DevOps Engineer,I'm always eager to learn and embrace change a...,"['flexible', 'change']"
1,Describe a time you handled a difficult situat...,Product Manager,"When faced with conflict, I approach it calmly...","['problem', 'disagree']"
2,Where do you see yourself in 5 years?,Data Scientist,My long-term goal is to grow into a senior rol...,"['aspiration', 'dream']"
3,Tell me about a conflict you had with a cowork...,Marketing Associate,"When faced with conflict, I approach it calmly...","['problem', 'disagree']"
4,What skills do you hope to develop in your nex...,Marketing Associate,My long-term goal is to grow into a senior rol...,"['future', 'aspiration']"


In [104]:
behavioral_questions = []

In [105]:
for index, row in df_test.iterrows():
    print('question : ',row['question'])
    result = classifier(row['question'])[0]
    print(f"  Predicted : {result['label']:<15} ({result['score']:.3f})")
    if (result['label'])=="BEHAVIORAL":
        behavioral_questions.append(row)
    print('answer : ',row['ideal_answer'])

question :  Tell me about a time you had to learn something completely new quickly.
  Predicted : BEHAVIORAL      (0.320)
answer :  I'm always eager to learn and embrace change as a way to improve. For example, I once had to switch to a new tech stack and picked it up quickly.
question :  Describe a time you handled a difficult situation professionally.
  Predicted : BEHAVIORAL      (0.428)
answer :  When faced with conflict, I approach it calmly and try to understand all perspectives before finding a solution together.
question :  Where do you see yourself in 5 years?
  Predicted : WARM_UP         (0.269)
answer :  My long-term goal is to grow into a senior role where I can contribute more strategically and mentor others. This role brings me closer to that.
question :  Tell me about a conflict you had with a coworker and how you resolved it.
  Predicted : BEHAVIORAL      (0.428)
answer :  When faced with conflict, I approach it calmly and try to understand all perspectives before find

In [108]:
# Convert the list of Series back into a DataFrame
behavioral_df = pd.DataFrame(behavioral_questions).reset_index(drop=True)

In [109]:
behavioral_df

,question,role,ideal_answer,keywords
0,Tell me about a time you had to learn somethin...,DevOps Engineer,I'm always eager to learn and embrace change a...,"['flexible', 'change']"
1,Describe a time you handled a difficult situat...,Product Manager,"When faced with conflict, I approach it calmly...","['problem', 'disagree']"
2,Tell me about a conflict you had with a cowork...,Marketing Associate,"When faced with conflict, I approach it calmly...","['problem', 'disagree']"


In [110]:
# 2. Construct a test case
for index, row in behavioral_df.iterrows():
    print(row['question'])
    print(row['ideal_answer'])

Tell me about a time you had to learn something completely new quickly.
I'm always eager to learn and embrace change as a way to improve. For example, I once had to switch to a new tech stack and picked it up quickly.
Describe a time you handled a difficult situation professionally.
When faced with conflict, I approach it calmly and try to understand all perspectives before finding a solution together.
Tell me about a conflict you had with a coworker and how you resolved it.
When faced with conflict, I approach it calmly and try to understand all perspectives before finding a solution together.


In [111]:
# 2. Construct a test case
test_case = LLMTestCase(
    input='Tell me about a time you had to learn something completely new quickly.',
    actual_output="I'm always eager to learn and embrace change as a way to improve. For example, I once had to switch to a new tech stack and picked it up quickly."
)

# 3. Run the evaluation
behavioral.measure(test_case)
print(behavioral.score)   # 0-1
print(behavioral.reason)  # Why this score was given

d:\Rushal\VS_code\Python\LocalAis\OLLAMA\ProjectIII\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

0.4
The response is vague and lacks a specific Situation or Task, as well as a clear Action taken and Result. The candidate's generic statement about being eager to learn and embrace change does not provide enough detail to meet the evaluation steps.


In [112]:
# 2. Construct a test case
for index, row in behavioral_df.iterrows():
    test_case = LLMTestCase(
        input=row['question'],
        actual_output=row['ideal_answer']
    )

    # 3. Run the evaluation
    behavioral.measure(test_case)
    print(behavioral.score)   # 0-1
    print(behavioral.reason)  # Why this score was given

d:\Rushal\VS_code\Python\LocalAis\OLLAMA\ProjectIII\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

0.4
The response is vague and lacks a specific Situation or Task, as well as a clear Action taken and Result. The candidate's generic statement about being eager to learn and embrace change does not provide enough detail to meet the evaluation steps.


0.4
The response is vague and lacks personal experience, failing to meet the evaluation steps' requirements for a specific situation, personal action, and outcome.


0.4
The response is vague and lacks personal experience, failing to meet the evaluation steps' requirements for a specific Situation, Action, and Result. The candidate's approach to conflict resolution is generic and lacks specific details.
